# Lab A — One Target, Several Monte Carlo Methods

**September 29, 2026 · Advanced Machine Learning · CPU runtime.**

Save a copy in Drive, then run from top to bottom. Read each prediction before running its experiment.
Setup checks package versions and verifies downloaded source hashes. If it asks for a restart, restart
the session once and run all cells again. No manual dataset upload, account key, or GPU is needed.
All data are transparently generated synthetic examples.

Exercises are independent: an unfinished response must not prevent later demonstrations.
Use the source links before calling an unfamiliar API. A permanent notebook link is different from
a temporary Gradio share URL, which requires an active runtime.


[Main lesson](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/course/notion/w5/w5-monte-carlo-inference.md) · [Repair note](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/course/notion/w5/w5-inference-repair.md) · [Full API guide](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/course/notion/w5/w5-code-guide.md)

We estimate the same first and second moments using iid MC, SNIS, MH, and Gibbs. The exact Gaussian is a teaching oracle. A method that works on this target may fail on another geometry.


In [ ]:
import sys, subprocess, importlib.metadata, hashlib, urllib.request, time
from pathlib import Path

PINS = {"numpy":"2.2.6", "matplotlib":"3.10.3", "scipy":"1.16.3",
        "pandas":"2.3.3", "arviz":"0.22.0", "gradio":"6.27.0"}
missing = []
for package, version in PINS.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(f"{package}=={version}")
restart = [p for p,v in PINS.items() if p in sys.modules and getattr(sys.modules[p], "__version__", v) != v]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
if restart:
    raise RuntimeError("Packages updated. Use Runtime > Restart session, then Run all. Loaded old packages: "+", ".join(restart))

CODE_REF = "2026-fall-w5"
EXPECTED = {'advml_vi.py': '699c3e5f7f75bee9dfdceff05e4956f7bd2bf0b9abff6c0eacf11a9ac3f1ad2a', 'advml_sampling.py': '7b881dd17a5895eb6c44835ae6e6516e2448258f99a76965a6b9d352049b0f1e', 'advml_smc.py': '890d797f1adaf009a5d46bf36058c5b8306654abfc6228412426486dc302deaa', 'advml_sampling_app.py': 'd4bfb72bb53c3c18908eb87e513bbf7a9cd5de050c0e4db8332948409b8a62ad'}
local = Path.cwd() / "src"
local_ok = all((local/name).is_file() and hashlib.sha256((local/name).read_bytes()).hexdigest()==digest
               for name,digest in EXPECTED.items())
support = local if local_ok else Path.cwd() / "w5_support" / CODE_REF
support.mkdir(parents=True, exist_ok=True)
downloaded = []
for name,digest in EXPECTED.items():
    path = support/name
    if path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest()==digest:
        continue
    url = f"https://raw.githubusercontent.com/lunalab-ai/advML/{CODE_REF}/src/{name}"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url, timeout=40) as response:
                payload = response.read()
            break
        except (OSError, TimeoutError):
            if attempt == 2:
                raise
            time.sleep(attempt+1)
    if hashlib.sha256(payload).hexdigest()!=digest:
        raise RuntimeError(f"Source hash mismatch: {name}. Do not run unverified code.")
    temporary = path.with_suffix(".download")
    temporary.write_bytes(payload)
    temporary.replace(path)
    downloaded.append(name)
sys.path.insert(0, str(support.resolve()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from IPython.display import display
print({"Python":sys.version.split()[0], "code_ref":CODE_REF, "downloaded":downloaded,
       "cache_or_reviewed_local":not downloaded, "versions":PINS})


## 1. Reuse the W4 target

### [make_target](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling.py#L82)

Build the W4 Gaussian posterior or a deliberately difficult mixture.

rho is the W4 OBSERVATION-noise correlation, not posterior correlation.
gaussian_model validates -1 < rho < 1. The mixture ignores rho and has
exact mean [0,0], covariance diag(9.25,.25). No fitting or random draws.
Example: make_target().mean is [5/6,-5/6].

### [Target.sample](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling.py#L67)

Return n independent oracle draws, shape (n,2), using local seed.

For the mixture each row chooses its own component. This is an exact
benchmark sampler, not an MCMC transition or a weighted sample.

### [mean_field_optimum](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_vi.py#L63)

Return the reverse-KL-optimal diagonal Gaussian (mean, variance).

Both outputs have shape (2,). The mean is the exact posterior mean;
variance[j] = 1/precision[j,j], generally NOT covariance[j,j].
Inputs are unchanged; no iteration or download occurs.


In [ ]:
from advml_sampling import (make_target, importance_sampling, normalize_log_weights,
    weight_ess, snis_summary, random_walk_mh, gaussian_gibbs, chain_diagnostics,
    mh_transition_matrix, autocorrelation)
from advml_vi import mean_field_optimum
target = make_target("gaussian", rho=.8)
exact_mean = target.mean[0]
exact_second = target.covariance[0,0] + exact_mean**2
vi_mean, vi_var = mean_field_optimum(target.model)
display(pd.DataFrame({"quantity":["E[z0]","Var[z0]","E[z0²]"],
                      "exact":[exact_mean,target.covariance[0,0],exact_second],
                      "mean_field":[vi_mean[0],vi_var[0],vi_var[0]+vi_mean[0]**2]}))


![Same target, different representations](../../course/notion/w5/assets/01-representations.png)

The mean-field mean is correct here, but its variance is too small. A first-moment check alone cannot find that error.


## A1 — your turn

Predict how the MCSE changes from N=100 to N=400. Then compare a repeated-seed RMSE with the error in one run. Does each individual run have to improve?


In [ ]:
# Write your experiment or calculation here. Later demonstrations do not depend on this cell.


In [ ]:
records=[]
for n in (25,100,400,1600):
    means=np.array([target.sample(n,seed)[:,0].mean() for seed in range(150)])
    records.append({"N":n, "posterior_SD":np.sqrt(target.covariance[0,0]),
                    "theoretical_MCSE":np.sqrt(target.covariance[0,0]/n),
                    "repeated_run_RMSE":np.sqrt(np.mean((means-exact_mean)**2)),
                    "seed_0_error":abs(means[0]-exact_mean)})
mc_table=pd.DataFrame(records);display(mc_table)
ax=mc_table.plot(x="N",y=["theoretical_MCSE","repeated_run_RMSE"],logx=True,logy=True,marker="o")
ax.set_ylabel("Numerical error scale");plt.show()


## 2. Importance sampling

### [importance_sampling](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling.py#L126)

Draw iid Gaussian proposals and return direct-IS/SNIS building blocks.

q has mean target.mean + [proposal_shift,0] and covariance
proposal_scale**2 * target.covariance. Scale is a SD multiplier (>0),
shift is in coordinate units. Return samples (n,2), normalized weights
(n,), log_weights (n,), weight_ess, log_normalizer_estimate, and evaluation
counts. Both benchmark targets have support R^2; this does not guarantee
finite variance for a narrow q. No support-coverage test is inferred from ESS.

### [normalize_log_weights](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling.py#L98)

Normalize a nonempty 1D log-weight array stably; return sum-one weights.

Minus infinity is allowed (zero weight); NaN, plus infinity and all-zero
mass raise ValueError. A common finite offset changes no normalized weight.
Example: log([1,2,7]) gives [.1,.2,.7]. No in-place mutation.

### [weight_ess](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling.py#L112)

Return 1/sum(normalized_weights**2), a WEIGHT-concentration diagnostic.

Accept nonnegative finite 1D weights with positive sum; normalize internally.
This is not MCMC autocorrelation ESS and cannot detect unsampled regions.
Uniform N weights yield N; a single nonzero weight yields 1.

### [snis_summary](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling.py#L152)

Summarize a scalar function on iid proposals with normalized IS weights.

values/weights are length n>=2. Return estimate, weighted posterior SD,
and plug-in delta-method MCSE sqrt(n/(n-1)*sum(W^2*(f-estimate)^2)).
MCSE is asymptotic, requires suitable finite moments and adequate coverage;
a small reported value is not a certificate. Not for correlated MCMC draws.


## A2 — your turn

For function values (0,1,2) and unnormalized weights (1,2,7), calculate the SNIS estimate and weight ESS by hand. Then write a stable normalization function for log weights shifted by +1000. Explain why exponentiating first can overflow.


In [ ]:
# Write your experiment or calculation here. Later demonstrations do not depend on this cell.


In [ ]:
is_rows=[]
for proposal_scale,proposal_shift in [(1.,0.),(1.7,0.),(.35,1.)]:
    result=importance_sampling(target,3000,proposal_scale,proposal_shift,seed=12)
    summary=snis_summary(result["samples"][:,0],result["weights"])
    is_rows.append({"scale":proposal_scale,"shift":proposal_shift,
                    "estimate":summary["estimate"],"error":abs(summary["estimate"]-exact_mean),
                    "weight_ESS":result["weight_ess"],"asymptotic_MCSE":summary["mcse"]})
display(pd.DataFrame(is_rows))


The Gaussian proposal covers the whole plane, even when narrow. Coverage is not a finite-variance guarantee. Inspect both error and weight concentration; a tiny reported MCSE can also be misleading when important regions were missed.


## 3. MH: include residence time

### [random_walk_mh](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling.py#L172)

Run Gaussian random-walk Metropolis; retain EVERY rejected state.

log_density maps a finite vector (d,) to a scalar log target up to a
constant. initial is a finite vector with finite log density; scale is
proposal SD in each coordinate. draws counts transitions, excludes the
initial state, and includes warmup if requested by the caller. Return
samples (draws,d), accepted bool (draws,), initial copy, target_evaluations
draws+1. No adaptation, thinning or automatic convergence claim.

### [mh_transition_matrix](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling.py#L277)

Exact finite-state MH transition matrix, including reject/self mass.

probabilities is a positive length-K target (normalized internally).
proposal is a KxK row-stochastic matrix q[j|i]. Return P[i,j]. Reverse
proposal probabilities enter the ratio; q[i,j]=0 permits no such move.
This small exact oracle supports the detailed-balance hand calculation.

### [Target.log_density](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling.py#L51)

Normalized log density at z of shape (...,2); return shape (...,).

A single length-2 vector returns a NumPy scalar. Natural logarithms.
This benchmark knows the normalizer; MH needs only a proportional
density. No samples or model state are changed by evaluation.


## A3 — your turn

A buggy MH loop appends a state only inside the accept branch. Describe the bias and repair the logic. For current=0, candidate=1, log_ratio=log(0.25), and u=0.70, what must be recorded? Check the finite three-state kernel against its target.


In [ ]:
# Write your experiment or calculation here. Later demonstrations do not depend on this cell.


## 4. Compare settings without hiding costs

### [static_experiment](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling_app.py#L13)

Run a fixed-target experiment and return real estimates plus raw draws.

method: 'IID MC', 'SNIS', 'MH', or 'Gibbs'. kind: gaussian/bimodal.
draws>=100 counts total proposals for IID/SNIS, POST-warmup draws PER CHAIN
for MH/Gibbs (four chains, 300 warmup transitions each). proposal_scale is
IS SD multiplier, proposal_shift shifts IS coordinate 0, mh_scale is MH
proposal SD. initial_center offsets coordinate 0 of dispersed starts.
seed is local; no learned state. Return target, samples (N,2), weights (N,),
chains (4,draws,2) or None, and scalar/JSON-ready metrics. Evaluation counts
include warmup; conditional draws and exact iid oracle calls are different
operations. Wall time is measured locally and is not a universal ranking.

### [chain_diagnostics](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling.py#L233)

ArviZ 0.22 diagnostics for one scalar observable, shape (chains,draws).

Supply >=2 chains, >=20 POST-warmup draws each, without flattening chains.
Return rank-normalized folded/split R-hat, bulk/tail/mean ESS, mean MCSE,
pooled mean/SD and retained count. Diagnostics concern the supplied f(z),
not every possible posterior quantity. Nonfinite diagnostic outputs become
None and an explicit warning; they are never replaced with a passing value.

### [autocorrelation](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling.py#L260)

Return the biased sample ACF at lags 0..max_lag for one 1D trace.

Used for visualization, not for computing ESS. Constant traces return NaN
beyond lag zero to avoid pretending that a stuck chain mixes perfectly.


In [ ]:
from advml_sampling_app import static_experiment, static_view
method_runs={method:static_experiment(method,draws=1000,seed=21)
             for method in ["IID MC","SNIS","MH","Gibbs"]}
columns=["method","retained_draws","estimate_z0","estimate_z0_squared",
         "mean_vector_error","covariance_error","mcse_z0","mcse_z0_squared",
         "target_evaluations","conditional_draws","iid_oracle_draws","seconds"]
comparison=pd.DataFrame([run["metrics"] for run in method_runs.values()])
display(comparison[columns])


IID/SNIS use 1,000 total draws here; MH/Gibbs use four chains of 1,000 retained draws, plus 300 warmup steps each. This table reports different operations; it does not assert equal computational budgets. Increase or constrain budgets explicitly if your question is efficiency.


## A4 — your turn

Change MH proposal SD across 0.03, 0.5, and 4.0. Predict acceptance and exploration separately. Use traces/ACF and at least one function-specific diagnostic to explain the result; do not select a winner using acceptance alone.


In [ ]:
# Write your experiment or calculation here. Later demonstrations do not depend on this cell.


In [ ]:
figure,diagnostics,csv_path=static_view("MH",draws=1000,mh_scale=.5,seed=22)
display(figure);display(diagnostics);plt.close(figure)
print("CSV record:",csv_path)


## 5. Gaussian Gibbs versus VI

### [gaussian_gibbs](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling.py#L208)

Sample exact Gaussian conditionals, coordinate 0 then coordinate 1.

draws counts complete sweeps. Conditional variance is 1/precision[j,j];
the mean uses the latest other coordinate. Return samples (draws,2),
substeps (2*draws+1,2), including initial, and conditional_draws=2*draws.
This is stochastic Gibbs, not deterministic CAVI or marginal iid sampling.


## A5 — your turn

The Gaussian Gibbs conditional variance is $1/P_{jj}$. Does that imply the marginal variance of the resulting chain is also $1/P_{jj}$? Explain the difference from the W4 mean-field optimum using an actual joint covariance estimate.


In [ ]:
# Write your experiment or calculation here. Later demonstrations do not depend on this cell.


In [ ]:
gibbs=gaussian_gibbs(target.model,[0,0],30,51)
path=gibbs["substeps"]
plt.figure(figsize=(6,4));plt.plot(path[:,0],path[:,1],"-o",markersize=2)
plt.xlabel("z0");plt.ylabel("z1");plt.title("Actual coordinate draws, not a scripted zigzag");plt.show()


## 6. Optional failure case: agreement inside one mode

The equal mixture has modes near z0=-3 and z0=3 and exact mean zero. All chains below start in the right mode. The target remains unchanged during the run.


In [ ]:
mixture=make_target("bimodal")
same_mode=np.stack([random_walk_mh(mixture.log_density,[3,0],3000,.3,120+j)["samples"][500:,0]
                    for j in range(4)])
mode_report=chain_diagnostics(same_mode)
display(mode_report)
print("estimated mean:",same_mode.mean(),"exact mean:",mixture.mean[0])


## A6 — your turn

Explain how these chains can have a near-one R-hat and a badly wrong mean. Propose a next experiment, and state why a nearly correct pooled mean from chains split across modes still need not prove mixing.


In [ ]:
# Write your experiment or calculation here. Later demonstrations do not depend on this cell.


## 7. Assemble a small interface

### [static_view](https://github.com/lunalab-ai/advML/blob/2026-fall-w5/src/advml_sampling_app.py#L107)

Gradio callback: same controls as static_experiment -> Figure, dict, CSV path.

Computes a NEW seeded experiment each time; writes one summary CSV in a
fresh temporary folder for download. Plots include exact target contours,
estimator/trace, weights/ACF and the two distinct expectation estimates.
No server is launched. Returned Figure belongs to the caller.

The slider value is passed to the callback. The callback fixes the other settings and returns a Figure, dictionary, and CSV path. The three output components must use that same order.


In [ ]:
def demo_callback(step_size):
    return static_view("MH",draws=500,mh_scale=float(step_size),seed=7)
with gr.Blocks(analytics_enabled=False) as small_app:
    gr.Markdown("# MH step-size laboratory")
    step_control=gr.Slider(.02,4,value=.5,label="Proposal SD")
    run_button=gr.Button("Run MH")
    picture=gr.Plot();numbers=gr.JSON();download=gr.File()
    run_button.click(demo_callback,[step_control],[picture,numbers,download])


## A7 — your turn

Extend the callback to return a fourth output: a sentence reporting the absolute error in E[z0²]. Add a Markdown output component and connect the four outputs in the correct order. Test the callback before launching.


In [ ]:
# Write your experiment or calculation here. Later demonstrations do not depend on this cell.


In [ ]:
assert np.isfinite(comparison["mean_vector_error"]).all()
assert target.sample(5,3).shape==(5,2)
np.testing.assert_allclose(normalize_log_weights(np.log([1,2,3])+700),np.array([1,2,3])/6)
check=random_walk_mh(target.log_density,[0,0],100,3.,77)
previous=np.vstack([check["initial"],check["samples"][:-1]])
np.testing.assert_array_equal(check["samples"][~check["accepted"]],previous[~check["accepted"]])
fig,metrics,path=demo_callback(.5)
assert len(fig.axes)==4 and pd.read_csv(path).shape[0]==1
plt.close(fig)
print("Lab A numerical and callback checks passed.")


Run the next cell in Colab to open the small app. The local validation path tests callbacks above without opening a public tunnel. Continue with [Lab B](https://colab.research.google.com/github/lunalab-ai/advML/blob/2026-fall-w5/notebooks/student/w5-b-sequential-monte-carlo.ipynb) for sequential filtering and the complete two-tab Sampling Explorer.


In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    small_app.launch(share=True,inline=True,prevent_thread_lock=True)
else:
    print("Interface assembled. For local interactive use: small_app.launch()")
